Simple notebook to test fitting using VecSpectrumDatasetOnOff that uses a vectorized version of the evaluator.

In [11]:
import numpy as np
import astropy.units as u
from gammapy.datasets import Datasets, DATASET_REGISTRY
from gammapy.utils.scripts import read_yaml, write_yaml, to_yaml
from gammapy.modeling.models import Models, GaussianPrior
from src.datasets import VecSpectrumDatasetOnOff

DATASET_REGISTRY.append(VecSpectrumDatasetOnOff)


First we modify the yaml file to read datasets as vectorized version

In [ ]:
# Modify datasets file
ds_dict = read_yaml("data/datasets.yaml")

for ds in ds_dict["datasets"]:
    ds['type'] = "VecSpectrumDatasetOnOff"
write_yaml(to_yaml(ds_dict), "data/datasets_vec.yaml", overwrite=True)

Now read the datasets.

In [3]:
datasets = Datasets.read("data/datasets_vec.yaml", filename_models="data/models.yaml")

/Users/terrier/Code/gammapy-dev/gammapy/gammapy/utils/scripts.py:73: UserWarning: Checksum verification failed.
  warnings.warn("Checksum verification failed.", UserWarning)
/Users/terrier/Code/gammapy-dev/gammapy/gammapy/utils/scripts.py:73: UserWarning: Checksum verification failed.
  warnings.warn("Checksum verification failed.", UserWarning)


For some unknown reasons, we lost the parameter linking and freezing. Reset it.

In [4]:
ref_par = datasets[0].models.parameters["alpha_norm"]
for ds in datasets[1:]:
    ds.models[0].spectral_model.model2.alpha_norm = ref_par 
    ds.models[0].spectral_model.model1.index2.frozen = True

Apply a prior on every free parameter to generate samples from.

In [5]:
for par in datasets.models.parameters.free_unique_parameters:
    par.prior = GaussianPrior(mu=par.value, sigma=0.2*par.value)

Reuse/adapt prior_inverse_cdf function from `Sampler`. Main changes:
- random variable generation is applied in the function
- return quantities instead of regular arrays as evaluation requires quantities

In [6]:
def prior_inverse_cdf(parameters, n_samples):
    """Returns a list of model parameters for a given list of values (that are bound in [0,1])."""
    from scipy.stats import uniform
    values = uniform(0,1).rvs(size=(len(parameters), n_samples)) 
    return [par.prior._inverse_cdf(val)*par.unit for par, val in zip(parameters, values)]

Take 100 samples.

In [7]:
samples = prior_inverse_cdf(datasets.models.parameters.free_unique_parameters, 100)
print(len(samples))

21


Now the vectorized npred evaluation function. We rely on a mapping between parameters and arrays of sample values.

In [8]:
def compute_npred_datasets(datasets, args):
    n_values = len(args[0])
    free_parameters = datasets.models.parameters.free_unique_parameters

    parameters = datasets.models.parameters.unique_parameters
    frozen_parameters = set(parameters)-set(free_parameters)

    index_arguments = {}
    for par, arg in zip(free_parameters, args):
        index_arguments[par] = arg

    for par in frozen_parameters:
        index_arguments[par] = np.ones(shape=(n_values))*par.quantity

    npreds = []
    for ds in datasets:
        vals = np.zeros((*ds.counts.data.shape[::-1], n_values))
        for key, eval in ds.evaluators.items():
            eval_args = [index_arguments[par] for par in eval.model.parameters]
            vals += eval.evaluate(*eval_args).to_value("")
        npreds.append(vals)
    return npreds


In [9]:
compute_npred_datasets(datasets, samples)

[array([[[[0.        , 0.        , 0.        , ..., 0.        ,
           0.        , 0.        ],
          [0.        , 0.        , 0.        , ..., 0.        ,
           0.        , 0.        ],
          [0.        , 0.        , 0.        , ..., 0.        ,
           0.        , 0.        ],
          ...,
          [0.31092553, 1.56342671, 1.1042625 , ..., 0.60780293,
           0.37791736, 0.12005601],
          [0.06212959, 0.34120672, 0.24622646, ..., 0.12656689,
           0.07470814, 0.02323041],
          [0.00987146, 0.05663799, 0.04086538, ..., 0.02036703,
           0.01186052, 0.00366123]]]], shape=(1, 1, 15, 100)),
 array([[[[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
           0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
          [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
           0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
          [1.01609217e-05, 2.72670575e-06, 8.76789082e-06, ...,
           3.30316955e-06, 3.09273084e-06, 1.8